# data import

In [1]:
import pandas as pd
from tqdm import tqdm
from os.path  import join

In [2]:
data_path = "../../data"

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

In [3]:
games = MRegularSeasonDetailedResults[['Season', 'DayNum', 'WTeamID', 'LTeamID']].copy().drop_duplicates()

# Rolling stats computation

In [4]:
def compute_rolling_stats(match_df, Team_id, Season, DayNum, n_matches=5):
    stat_columns = [
        'Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 
        'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF'
    ]
    
    # Filter matches where the team was involved
    match_df = match_df[(match_df['WTeamID'] == Team_id) | (match_df['LTeamID'] == Team_id)]
    
    # Sort by Season and DayNum
    match_df = match_df.sort_values(['Season', 'DayNum'])
    
    # Filter only matches before the given Season and DayNum
    match_df = match_df[(match_df['Season'] < Season) | ((match_df['Season'] == Season) & (match_df['DayNum'] < DayNum))]
    
    # Take the last `n_matches`
    last_n_matches = match_df.tail(n_matches).copy()  # Copy to avoid modifying the original dataframe

    for col in stat_columns:
        last_n_matches[col] = last_n_matches.apply(
            lambda row: row[f'W{col}'] if row['WTeamID'] == Team_id else row[f'L{col}'], axis=1
        )

    # Compute rolling averages for the selected matches
    rolling_stats = last_n_matches[stat_columns].mean()

    return last_n_matches, rolling_stats

# # Example call
# last_matches, rolling_stats = compute_rolling_stats(df, 1104, 2020, 0)
# print(last_matches)
# print(rolling_stats)


# Target computation

In [5]:
data_path = '../../data'

# The Basics ------------------------------------------------------------------------
# Men
MTeams = pd.read_csv(join(data_path, 'MTeams.csv'))
MSeasons = pd.read_csv(join(data_path, 'MSeasons.csv'))
MNCAATourneySeeds = pd.read_csv(join(data_path, 'MNCAATourneySeeds.csv'))
MRegularSeasonCompactResults = pd.read_csv(join(data_path, 'MRegularSeasonCompactResults.csv'))
MNCAATourneyCompactResults = pd.read_csv(join(data_path, 'MNCAATourneyCompactResults.csv'))
# Women
WTeams = pd.read_csv(join(data_path, 'WTeams.csv'))
WSeasons = pd.read_csv(join(data_path, 'WSeasons.csv'))
WNCAATourneySeeds = pd.read_csv(join(data_path, 'WNCAATourneySeeds.csv'))
WRegularSeasonCompactResults = pd.read_csv(join(data_path, 'WRegularSeasonCompactResults.csv'))
WNCAATourneyCompactResults = pd.read_csv(join(data_path, 'WNCAATourneyCompactResults.csv'))
# Other
SampleSubmissionStage1 = pd.read_csv(join(data_path, 'SampleSubmissionStage1.csv'))
SampleSubmissionStage2 = pd.read_csv(join(data_path, 'SampleSubmissionStage2.csv'))
SeedBenchmarkStage1 = pd.read_csv(join(data_path, 'SeedBenchmarkStage1.csv'))

# Team Box Scores ------------------------------------------------------------------------
# Men
MRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'MRegularSeasonDetailedResults.csv'))
MNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'MNCAATourneyDetailedResults.csv'))
# Women
WRegularSeasonDetailedResults = pd.read_csv(join(data_path, 'WRegularSeasonDetailedResults.csv'))
WNCAATourneyDetailedResults = pd.read_csv(join(data_path, 'WNCAATourneyDetailedResults.csv'))

# Geography ------------------------------------------------------------------------
# All
Cities = pd.read_csv(join(data_path, 'Cities.csv'))
Conferences = pd.read_csv(join(data_path, 'Conferences.csv'))
# Men
MGameCities = pd.read_csv(join(data_path, 'MGameCities.csv'))
# Women
WGameCities = pd.read_csv(join(data_path, 'WGameCities.csv'))

# Public Rankings ------------------------------------------------------------------------
# Men
MMasseyOrdinals = pd.read_csv(join(data_path, 'MMasseyOrdinals.csv')) # men only

# Supplements ------------------------------------------------------------------------
# Men
MTeamCoaches = pd.read_csv(join(data_path, 'MTeamCoaches.csv')) # men only
MTeamConferences = pd.read_csv(join(data_path, 'MTeamConferences.csv'))
MConferenceTourneyGames = pd.read_csv(join(data_path, 'MConferenceTourneyGames.csv'))
MSecondaryTourneyTeams = pd.read_csv(join(data_path, 'MSecondaryTourneyTeams.csv'))
MSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'MSecondaryTourneyCompactResults.csv'))
MTeamSpellings = pd.read_csv(join(data_path, "MTeamSpellings.csv"), encoding='cp1252')
MNCAATourneySlots = pd.read_csv(join(data_path, 'MNCAATourneySlots.csv'))
MNCAATourneySeedRoundSlots = pd.read_csv(join(data_path, 'MNCAATourneySeedRoundSlots.csv')) # men only
# Women
WTeamConferences = pd.read_csv(join(data_path, 'WTeamConferences.csv'))
WConferenceTourneyGames = pd.read_csv(join(data_path, 'WConferenceTourneyGames.csv'))
WSecondaryTourneyTeams = pd.read_csv(join(data_path, 'WSecondaryTourneyTeams.csv'))
WSecondaryTourneyCompactResults = pd.read_csv(join(data_path, 'WSecondaryTourneyCompactResults.csv'))
WTeamSpellings = pd.read_csv(join(data_path, 'WTeamSpellings.csv'), encoding='cp1252')
WNCAATourneySlots = pd.read_csv(join(data_path, 'WNCAATourneySlots.csv'))

In [6]:
# regular_results = pd.concat([
#     MRegularSeasonDetailedResults.copy(),
#     WRegularSeasonDetailedResults.copy()
# ], ignore_index=True)
# tourney_results = pd.concat([
#     MNCAATourneyDetailedResults.copy(),
#     WNCAATourneyDetailedResults.copy()
# ], ignore_index=True)

In [7]:
# regular_results

In [8]:
# rolling_stats_results = []
# for _, row in tqdm(games.iterrows(), total=len(games), desc="Processing Games"):
#     season, daynum, wteam, lteam = row['Season'], row['DayNum'], row['WTeamID'], row['LTeamID']
    
#     wteam_stats = compute_rolling_stats(df, wteam, season, daynum)
#     lteam_stats = compute_rolling_stats(df, lteam, season, daynum)

#     rolling_stats_results.append({'Season': season, 'DayNum': daynum, 'TeamID': wteam, 'RollingStats': wteam_stats})
#     rolling_stats_results.append({'Season': season, 'DayNum': daynum, 'TeamID': lteam, 'RollingStats': lteam_stats})

# rolling_stats_df = pd.DataFrame(rolling_stats_results)

# V2

In [9]:
# import os


# os.environ["MODIN_ENGINE"] = "dask"  # Modin will use Ray

# import modin.pandas as pd

In [10]:
season_year = 2024

In [11]:
def create_features_and_target(games_df, n_matches=5):
    # List of stat columns (without the W/L prefixes)
    stats = ['Score', 'FGM', 'FGA', 'FGM3', 'FGA3', 
             'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF']

    # --- Build a long DataFrame with one row per team-game ---
    # For winning teams (label 1)
    w_df = games_df[['Season', 'DayNum', 'WTeamID'] + [f'W{stat}' for stat in stats]].copy()
    w_df.rename(columns={'WTeamID': 'TeamID'}, inplace=True)
    for stat in stats:
        w_df[stat] = w_df[f'W{stat}']
    w_df['Won'] = 1
    w_df = w_df[['Season', 'DayNum', 'TeamID'] + stats + ['Won']]
    
    # For losing teams (label 0)
    l_df = games_df[['Season', 'DayNum', 'LTeamID'] + [f'L{stat}' for stat in stats]].copy()
    l_df.rename(columns={'LTeamID': 'TeamID'}, inplace=True)
    for stat in stats:
        l_df[stat] = l_df[f'L{stat}']
    l_df['Won'] = 0
    l_df = l_df[['Season', 'DayNum', 'TeamID'] + stats + ['Won']]
    
    # Combine winners and losers into one DataFrame
    combined = pd.concat([w_df, l_df], ignore_index=True)
    combined.sort_values(['TeamID', 'Season', 'DayNum'], inplace=True)
    combined.reset_index(drop=True, inplace=True)
    
    # --- Compute rolling averages for the stats ---
    # For each team, compute rolling averages over the previous n_matches
    rolling = combined.groupby('TeamID')[stats].apply(
        lambda group: group.shift(1).rolling(window=n_matches, min_periods=1).mean()
    ).reset_index(drop=True)
    rolling = rolling.add_suffix('_rolling')
    
    # Combine the rolling averages with the original DataFrame
    combined_stats = pd.concat([combined, rolling], axis=1)
    
    # --- Separate the features and target ---
    # Now include Season, DayNum, and TeamID in the features
    feature_cols = ["Season", "DayNum", "TeamID"] + [f"{stat}_rolling" for stat in stats]
    X = combined_stats[feature_cols].copy()
    y = combined_stats['Won'].copy()
    
    return X, y, combined_stats

def get_teamid_in_tournament(tournament_results, year):
    tournament_results = tournament_results[tournament_results['Season'] == year]
    teams = set(tournament_results['WTeamID']).union(set(tournament_results['LTeamID']))
    return teams

def filter_stats(combined_stats, teams_in_tourney):
    return combined_stats[combined_stats['TeamID'].isin(teams_in_tourney)]

def create_X_and_y(combined_stats, teams_in_tourney, regular_results):
    filtered_stats = filter_stats(combined_stats, teams_in_tourney)
    games_with_teams_in_tournament = regular_results[
        regular_results['WTeamID'].isin(teams_in_tourney) | regular_results['LTeamID'].isin(teams_in_tourney)
    ]
    games_with_teams_in_tournament = games_with_teams_in_tournament[['Season', 'DayNum', 'WTeamID', 'LTeamID']]
    
    # Create a DataFrame where Team1 is the winner and Team2 is the loser
    games_with_teams_12 = games_with_teams_in_tournament.copy()
    games_with_teams_12 = games_with_teams_12.rename(columns={'WTeamID': 'Team1', 'LTeamID': 'Team2'})
    games_with_teams_12['Won'] = 1  # Team1 won

    # Create a DataFrame where Team1 is the loser and Team2 is the winner
    games_with_teams_21 = games_with_teams_in_tournament.copy()
    games_with_teams_21 = games_with_teams_21.rename(columns={'LTeamID': 'Team1', 'WTeamID': 'Team2'})
    games_with_teams_21['Won'] = 0  # Team1 lost
    
    # Combine both DataFrames
    games_combined = pd.concat([games_with_teams_12, games_with_teams_21], ignore_index=True)

    print(len(games_with_teams_in_tournament))
    print(len(games_combined))
    print(games_combined.columns)
    
    return games_combined




In [12]:
teams_in_tourney = get_teamid_in_tournament(MNCAATourneyCompactResults, 2024)
len(teams_in_tourney)

68

In [13]:
X, y, combined_stats = create_features_and_target(MRegularSeasonDetailedResults, n_matches=5)

In [14]:
print(combined_stats.columns)
combined_stats

Index(['Season', 'DayNum', 'TeamID', 'Score', 'FGM', 'FGA', 'FGM3', 'FGA3',
       'FTM', 'FTA', 'OR', 'DR', 'Ast', 'TO', 'Stl', 'Blk', 'PF', 'Won',
       'Score_rolling', 'FGM_rolling', 'FGA_rolling', 'FGM3_rolling',
       'FGA3_rolling', 'FTM_rolling', 'FTA_rolling', 'OR_rolling',
       'DR_rolling', 'Ast_rolling', 'TO_rolling', 'Stl_rolling', 'Blk_rolling',
       'PF_rolling'],
      dtype='object')


,Season,DayNum,TeamID,Score,FGM,FGA,FGM3,FGA3,FTM,FTA,...,FGA3_rolling,FTM_rolling,FTA_rolling,OR_rolling,DR_rolling,Ast_rolling,TO_rolling,Stl_rolling,Blk_rolling,PF_rolling
0,2014,5,1101,75,22,63,7,25,24,32,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2014,7,1101,47,14,40,4,16,15,26,...,25.000000,24.00,32.000000,11.000000,22.000000,9.000000,10.000000,3.000000,1.000000,26.000000
2,2014,9,1101,44,17,54,10,21,0,0,...,20.500000,19.50,29.000000,11.000000,19.500000,8.000000,14.000000,2.500000,1.000000,28.000000
3,2014,13,1101,41,14,54,3,23,10,12,...,20.666667,13.00,19.333333,9.333333,18.666667,8.333333,14.666667,3.666667,0.666667,23.666667
4,2014,15,1101,64,19,50,9,20,17,25,...,21.250000,12.25,17.500000,9.500000,19.000000,7.750000,15.750000,3.500000,0.500000,23.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235491,2025,93,1480,67,28,62,5,21,6,7,...,16.800000,16.00,20.000000,8.600000,21.000000,14.000000,10.600000,9.000000,3.400000,15.200000
235492,2025,96,1480,61,23,61,4,22,11,19,...,17.000000,12.40,15.800000,8.200000,21.600000,14.000000,10.800000,9.000000,3.400000,12.400000
235493,2025,101,1480,73,28,58,6,16,11,16,...,17.800000,13.20,17.400000,9.600000,22.000000,13.400000,9.200000,8.000000,3.400000,13.000000
235494,2025,103,1480,81,23,49,10,22,25,31,...,17.600000,11.00,15.600000,9.400000,21.200000,12.200000,9.400000,7.200000,3.000000,13.200000


In [15]:
pairings = create_X_and_y(combined_stats, teams_in_tourney, MRegularSeasonDetailedResults)

41027
82054
Index(['Season', 'DayNum', 'Team1', 'Team2', 'Won'], dtype='object')


In [16]:
def join_pairings_with_team_stats(pairings_l, combined_st, join_on_day = True, season = None):
    # Define the join keys
    if join_on_day:
        join_keys = ['Season', 'DayNum', 'TeamID']
        # Create a copy of combined_stats and add the prefix "team_1_" to non-key columns
        combined_stats_team1 = combined_st.copy()
        cols_to_prefix = [col for col in combined_stats_team1.columns if col not in join_keys]
        combined_stats_team1 = combined_stats_team1.rename(
            columns={col: f"team_1_{col}" for col in cols_to_prefix}
        )

        # Now merge pairings with the renamed combined_stats on Season, DayNum, and Team1 == TeamID
        pairings_combined_with_stats = pd.merge(
            pairings_l,
            combined_stats_team1,
            how='left',
            left_on=['Season', 'DayNum', 'Team1'],
            right_on=['Season', 'DayNum', 'TeamID']
        )

        # Create a copy of combined_stats and add the prefix "team_2_" to non-key columns
        combined_stats_team2 = combined_st.copy()
        combined_stats_team2 = combined_stats_team2.rename(
            columns={col: f"team_2_{col}" for col in cols_to_prefix}
        )

        # Now merge pairings_combined_with_stats with the renamed combined_stats on Season, DayNum, and Team2 == TeamID
        pairings_combined_with_stats = pd.merge(
            pairings_combined_with_stats,
            combined_stats_team2,
            how='left',
            left_on=['Season', 'DayNum', 'Team2'],
            right_on=['Season', 'DayNum', 'TeamID']
        )

        # Remove NaN values

        pairings_combined_with_stats.dropna(inplace=True)
        return pairings_combined_with_stats
    else:
        join_keys = ['Season', 'DayNum', 'TeamID']
        # print(combined_st.columns, combined_st.shape)
        # Filter for the given season and keep only the last record per team (by DayNum)
        combined_st = combined_st[combined_st['Season'] == season]
        combined_st = combined_st.sort_values('DayNum').drop_duplicates(subset='TeamID', keep='last')

        combined_stats_team1 = combined_st.copy()
        cols_to_prefix = [col for col in combined_stats_team1.columns if col not in ['Season', 'DayNum', 'TeamID']]
        combined_stats_team1 = combined_stats_team1.rename(
            columns={col: f"team_1_{col}" for col in cols_to_prefix}
        )        
        # print("Combinde", combined_stats_team1.columns, combined_stats_team1.shape)

        pairings_combined_with_stats = pd.merge(
            pairings_l,
            combined_stats_team1,
            how='left',
            left_on=['Season', 'Team1'],
            right_on=['Season', 'TeamID']
        )

        combined_stats_team2 = combined_st.copy()
        cols_to_prefix = [col for col in combined_stats_team2.columns if col not in ['Season', 'DayNum', 'TeamID']]
        combined_stats_team2 = combined_stats_team2.rename(
            columns={col: f"team_2_{col}" for col in cols_to_prefix}
        )        

        pairings_combined_with_stats = pd.merge(
            pairings_combined_with_stats,
            combined_stats_team2,
            how='left',
            left_on=['Season',  'Team2'],
            right_on=['Season', 'TeamID']
        )

        # Remove NaN values

        pairings_combined_with_stats.dropna(inplace=True)
        return pairings_combined_with_stats

    

USUN TU

DOTAD

In [17]:
pairings_combined_with_stats = join_pairings_with_team_stats(pairings, combined_stats)


In [18]:
nan_counts = pairings_combined_with_stats.isna().sum()
print(nan_counts)
print(pairings_combined_with_stats.shape)

Season                0
DayNum                0
Team1                 0
Team2                 0
Won                   0
                     ..
team_2_Ast_rolling    0
team_2_TO_rolling     0
team_2_Stl_rolling    0
team_2_Blk_rolling    0
team_2_PF_rolling     0
Length: 65, dtype: int64
(81868, 65)


In [19]:
print(pairings[(pairings['Season'] == 2024) & (pairings['DayNum'] == 132)])
print(MRegularSeasonDetailedResults[(MRegularSeasonDetailedResults['Season'] == 2024) & (MRegularSeasonDetailedResults['DayNum'] == 132)])

       Season  DayNum  Team1  Team2  Won
39546    2024     132   1120   1196    1
39547    2024     132   1182   1433    1
39548    2024     132   1228   1458    1
39549    2024     132   1412   1396    1
39550    2024     132   1463   1135    1
80573    2024     132   1196   1120    0
80574    2024     132   1433   1182    0
80575    2024     132   1458   1228    0
80576    2024     132   1396   1412    0
80577    2024     132   1135   1463    0
        Season  DayNum  WTeamID  WScore  LTeamID  LScore WLoc  NumOT  WFGM  \
113236    2024     132     1120      86     1196      67    N      0    31   
113237    2024     132     1182      57     1433      51    N      0    17   
113238    2024     132     1228      93     1458      87    N      0    30   
113239    2024     132     1412      85     1396      69    N      0    31   
113240    2024     132     1463      62     1135      61    N      0    25   

        WFGA  ...  LFGA3  LFTM  LFTA  LOR  LDR  LAst  LTO  LStl  LBlk  LPF  
113

In [20]:
def generate_X_y_from_pairings(pairings):
    return pairings.drop(columns=['Season', 'DayNum', 'Team1', 'Team2', 'Won']), pairings['Won']

In [21]:
X, y = generate_X_y_from_pairings(pairings_combined_with_stats)

In [22]:
X.columns

Index(['TeamID_x', 'team_1_Score', 'team_1_FGM', 'team_1_FGA', 'team_1_FGM3',
       'team_1_FGA3', 'team_1_FTM', 'team_1_FTA', 'team_1_OR', 'team_1_DR',
       'team_1_Ast', 'team_1_TO', 'team_1_Stl', 'team_1_Blk', 'team_1_PF',
       'team_1_Won', 'team_1_Score_rolling', 'team_1_FGM_rolling',
       'team_1_FGA_rolling', 'team_1_FGM3_rolling', 'team_1_FGA3_rolling',
       'team_1_FTM_rolling', 'team_1_FTA_rolling', 'team_1_OR_rolling',
       'team_1_DR_rolling', 'team_1_Ast_rolling', 'team_1_TO_rolling',
       'team_1_Stl_rolling', 'team_1_Blk_rolling', 'team_1_PF_rolling',
       'TeamID_y', 'team_2_Score', 'team_2_FGM', 'team_2_FGA', 'team_2_FGM3',
       'team_2_FGA3', 'team_2_FTM', 'team_2_FTA', 'team_2_OR', 'team_2_DR',
       'team_2_Ast', 'team_2_TO', 'team_2_Stl', 'team_2_Blk', 'team_2_PF',
       'team_2_Won', 'team_2_Score_rolling', 'team_2_FGM_rolling',
       'team_2_FGA_rolling', 'team_2_FGM3_rolling', 'team_2_FGA3_rolling',
       'team_2_FTM_rolling', 'team_2_FTA_ro

# Tournament data preperation

In [23]:
MRegularSeasonCompactResults.columns

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT'],
      dtype='object')

In [24]:
MNCAATourneyCompactResults.columns

Index(['Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc',
       'NumOT'],
      dtype='object')

In [25]:
tournament_pairings = create_X_and_y(combined_stats, teams_in_tourney, MNCAATourneyCompactResults)
tournament_pairings = tournament_pairings[tournament_pairings['Season']==season_year]
tournament_pairings

1621
3242
Index(['Season', 'DayNum', 'Team1', 'Team2', 'Won'], dtype='object')


,Season,DayNum,Team1,Team2,Won
1554,2024,134,1161,1438,1
1555,2024,134,1447,1224,1
1556,2024,135,1160,1129,1
1557,2024,135,1212,1286,1
1558,2024,136,1112,1253,1
...,...,...,...,...,...
3237,2024,146,1181,1301,0
3238,2024,146,1397,1345,0
3239,2024,152,1104,1163,0
3240,2024,152,1301,1345,0


In [26]:
regular_season_and_tournament_combined = pd.concat(
    [MRegularSeasonDetailedResults, MNCAATourneyDetailedResults],
    ignore_index=True
).copy()

join_pairings_with_team_stats(tournament_pairings, combined_stats, join_on_day=False, season=season_year)


,Season,DayNum_x,Team1,Team2,Won,DayNum_y,TeamID_x,team_1_Score,team_1_FGM,team_1_FGA,...,team_2_FGA3_rolling,team_2_FTM_rolling,team_2_FTA_rolling,team_2_OR_rolling,team_2_DR_rolling,team_2_Ast_rolling,team_2_TO_rolling,team_2_Stl_rolling,team_2_Blk_rolling,team_2_PF_rolling
0,2024,134,1161,1438,1,130,1161,61,24,70,...,20.6,9.0,14.4,7.6,24.4,13.0,6.2,5.2,5.2,11.8
1,2024,134,1447,1224,1,127,1447,54,22,49,...,19.8,18.2,23.8,9.4,22.2,15.2,12.0,5.6,2.8,19.6
2,2024,135,1160,1129,1,131,1160,68,20,48,...,22.8,18.4,23.8,11.6,24.6,12.4,10.0,6.2,2.2,20.0
3,2024,135,1212,1286,1,131,1212,75,23,47,...,27.2,10.6,14.4,6.4,23.8,15.4,10.6,8.2,4.0,15.2
4,2024,136,1112,1253,1,130,1112,59,18,47,...,18.0,17.6,25.0,12.6,23.4,16.4,10.6,6.4,4.2,17.4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
129,2024,146,1181,1301,0,129,1181,69,28,65,...,19.0,18.0,23.0,8.8,23.8,13.2,9.0,8.2,4.8,12.0
130,2024,146,1397,1345,0,130,1397,56,19,62,...,18.4,15.6,22.2,10.2,22.8,18.6,9.2,5.6,2.4,16.6
131,2024,152,1104,1163,0,130,1104,88,33,71,...,22.8,12.6,17.4,10.2,26.0,22.4,8.0,5.6,5.8,14.8
132,2024,152,1301,1345,0,131,1301,84,28,51,...,18.4,15.6,22.2,10.2,22.8,18.6,9.2,5.6,2.4,16.6


In [27]:
pairings_combined_with_stats_tournament = join_pairings_with_team_stats(pairings, combined_stats)
nan_counts = pairings_combined_with_stats_tournament.isna().sum()
print(nan_counts)
print(pairings_combined_with_stats_tournament.shape)


X_tournament, y_tournament = generate_X_y_from_pairings(pairings_combined_with_stats_tournament)

Season                0
DayNum                0
Team1                 0
Team2                 0
Won                   0
                     ..
team_2_Ast_rolling    0
team_2_TO_rolling     0
team_2_Stl_rolling    0
team_2_Blk_rolling    0
team_2_PF_rolling     0
Length: 65, dtype: int64
(81868, 65)


# Models

In [28]:
from os.path  import join
import random
import itertools
import math

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold
import statsmodels.api as sm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline
import xgboost as xgb
from xgboost import XGBClassifier
import optuna

## Basic LR

In [29]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegressionCV(
        cv=5,                # 5-fold cross-validation
        solver='lbfgs',
        max_iter=2000,       # Increase max_iter if needed
        scoring='neg_log_loss',  # Optimize log-loss for probability calibration
        refit=True
    ))
])

# Train the model using the pipeline
pipeline.fit(X, y)


Pipeline(steps=[('scaler', StandardScaler()),
                ('clf',
                 LogisticRegressionCV(cv=5, max_iter=2000,
                                      scoring='neg_log_loss'))])

In [30]:
X.columns

Index(['TeamID_x', 'team_1_Score', 'team_1_FGM', 'team_1_FGA', 'team_1_FGM3',
       'team_1_FGA3', 'team_1_FTM', 'team_1_FTA', 'team_1_OR', 'team_1_DR',
       'team_1_Ast', 'team_1_TO', 'team_1_Stl', 'team_1_Blk', 'team_1_PF',
       'team_1_Won', 'team_1_Score_rolling', 'team_1_FGM_rolling',
       'team_1_FGA_rolling', 'team_1_FGM3_rolling', 'team_1_FGA3_rolling',
       'team_1_FTM_rolling', 'team_1_FTA_rolling', 'team_1_OR_rolling',
       'team_1_DR_rolling', 'team_1_Ast_rolling', 'team_1_TO_rolling',
       'team_1_Stl_rolling', 'team_1_Blk_rolling', 'team_1_PF_rolling',
       'TeamID_y', 'team_2_Score', 'team_2_FGM', 'team_2_FGA', 'team_2_FGM3',
       'team_2_FGA3', 'team_2_FTM', 'team_2_FTA', 'team_2_OR', 'team_2_DR',
       'team_2_Ast', 'team_2_TO', 'team_2_Stl', 'team_2_Blk', 'team_2_PF',
       'team_2_Won', 'team_2_Score_rolling', 'team_2_FGM_rolling',
       'team_2_FGA_rolling', 'team_2_FGM3_rolling', 'team_2_FGA3_rolling',
       'team_2_FTM_rolling', 'team_2_FTA_ro

In [36]:
y_prob

array([9.99999421e-01, 9.99999143e-01, 9.99999686e-01, ...,
       1.10890019e-06, 2.52625905e-04, 1.69599473e-06], shape=(81868,))

In [37]:
y_tournament

6        1
8        1
9        1
13       1
14       1
        ..
82049    0
82050    0
82051    0
82052    0
82053    0
Name: Won, Length: 81868, dtype: int64

In [35]:
y_prob = pipeline.predict_proba(X_tournament)[:, 1]

score = brier_score_loss(y_tournament, y_prob)
print("Brier score:", score)

Brier score: 8.884160451976025e-09


In [ ]:
import pickle
from sklearn.linear_model import LinearRegression, LassoCV, LogisticRegression, LogisticRegressionCV
from sklearn.model_selection import train_test_split, TimeSeriesSplit, cross_val_score, StratifiedKFold

In [ ]:
pickle.dump(X, open("X.pkl", "wb"))
pickle.dump(y, open("y.pkl", "wb"))